In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [2]:
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

In [3]:
trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=train_transform
)
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=test_transform
)

batch_size = 128
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
testloader  = DataLoader(testset,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Train size:", len(trainset), "Test size:", len(testset))

Train size: 50000 Test size: 10000


In [4]:
images, labels = next(iter(trainloader))
print("Images:", images.shape, images.dtype)
print("Labels:", labels.shape, labels.dtype)
print("Label example:", labels[:10])

C:\Users\ben21\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Images: torch.Size([128, 3, 32, 32]) torch.float32
Labels: torch.Size([128]) torch.int64
Label example: tensor([9, 7, 6, 2, 8, 4, 7, 6, 8, 1])


In [5]:
class CNN3Blocks(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # 32x32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8x8

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 4x4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = CNN3Blocks().to(device)

In [6]:
images, labels = images.to(device), labels.to(device)
logits = model(images)
print("Logits shape:", logits.shape)


Logits shape: torch.Size([128, 10])


In [7]:

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,          
    momentum=0.9,
    weight_decay=5e-4  
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

loss = criterion(logits, labels)
print("One batch loss:", loss.item())


One batch loss: 2.361908197402954


In [8]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

In [9]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, total_correct / total

In [10]:
epochs = 15
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, trainloader)
    test_loss, test_acc = evaluate(model, testloader)

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:02d}/{epochs} | LR: {lr:.6f} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | "
          f"Test Loss: {test_loss:.4f} Acc: {test_acc*100:.2f}%")

    scheduler.step()

Epoch 01/15 | LR: 0.100000 | Train Loss: 1.9744 Acc: 24.57% | Test Loss: 1.6202 Acc: 39.57%
Epoch 02/15 | LR: 0.100000 | Train Loss: 1.5125 Acc: 44.61% | Test Loss: 1.2200 Acc: 54.26%
Epoch 03/15 | LR: 0.100000 | Train Loss: 1.2718 Acc: 54.97% | Test Loss: 1.1464 Acc: 58.27%
Epoch 04/15 | LR: 0.100000 | Train Loss: 1.1382 Acc: 60.23% | Test Loss: 1.4002 Acc: 57.38%
Epoch 05/15 | LR: 0.100000 | Train Loss: 1.0717 Acc: 62.47% | Test Loss: 0.9492 Acc: 67.68%
Epoch 06/15 | LR: 0.050000 | Train Loss: 0.9018 Acc: 68.78% | Test Loss: 0.8087 Acc: 72.31%
Epoch 07/15 | LR: 0.050000 | Train Loss: 0.8572 Acc: 70.45% | Test Loss: 0.7161 Acc: 75.02%
Epoch 08/15 | LR: 0.050000 | Train Loss: 0.8351 Acc: 71.35% | Test Loss: 0.7935 Acc: 73.52%
Epoch 09/15 | LR: 0.050000 | Train Loss: 0.8177 Acc: 71.94% | Test Loss: 0.7042 Acc: 76.24%
Epoch 10/15 | LR: 0.050000 | Train Loss: 0.7960 Acc: 72.88% | Test Loss: 0.9154 Acc: 69.17%
Epoch 11/15 | LR: 0.025000 | Train Loss: 0.6985 Acc: 76.00% | Test Loss: 0.7276 